In [29]:
from dataclasses import dataclass
import logging
from typing import Any, Dict
import pytest

# Logging Setup
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


# 1. Custom Exceptions
class WeatherServiceError(Exception):
    """Base exception for application errors."""

    pass


class InvalidLocationError(WeatherServiceError):
    """Raised when location is invalid or empty."""

    pass


class ExternalAPIError(WeatherServiceError):
    """Raised when data payload is malformed."""

    pass


# 2. Data Models
@dataclass(frozen=True)
class WeatherReport:
    """Dataclass storing immutable weather report details."""

    city: str
    temperature_celsius: float
    condition: str

    @property
    def temperature_fahrenheit(self) -> float:
        return (self.temperature_celsius * 9 / 5) + 32


# 3. Business Logic Processor
class WeatherProcessor:
    """Core processor for handling weather data."""

    def process_api_response(
        self, city: str, raw_data: Dict[str, Any]
    ) -> WeatherReport:
        if not city or not city.strip():
            logger.error("Invalid city provided.")
            raise InvalidLocationError("City name cannot be empty.")

        if "temp" not in raw_data or "status" not in raw_data:
            logger.error(f"Missing keys in payload for city {city}.")
            raise ExternalAPIError("Required keys missing in weather payload.")

        logger.info(f"Successfully processed report for: {city}")
        return WeatherReport(
            city=city.strip().capitalize(),
            temperature_celsius=float(raw_data["temp"]),
            condition=str(raw_data["status"]),
        )


print("✅ Core application classes loaded successfully!")

✅ Core application classes loaded successfully!


In [30]:
def run_app():
    processor = WeatherProcessor()

    # Valid Payload Demo
    sample_payload = {"temp": 28.5, "status": "Sunny"}

    try:
        report = processor.process_api_response("Lahore", sample_payload)
        print("\n--- Weather Summary ---")
        print(f"City: {report.city}")
        print(f"Celsius: {report.temperature_celsius}°C")
        print(f"Fahrenheit: {report.temperature_fahrenheit:.1f}°F")
        print(f"Condition: {report.condition}\n")
    except WeatherServiceError as err:
        print(f"Error: {err}")


run_app()


--- Weather Summary ---
City: Lahore
Celsius: 28.5°C
Fahrenheit: 83.3°F
Condition: Sunny



In [31]:
def test_process_valid_weather_data():
    processor = WeatherProcessor()
    payload = {"temp": 20.0, "status": "Rainy"}
    report = processor.process_api_response("faisalabad", payload)

    assert report.city == "Faisalabad"
    assert report.temperature_celsius == 20.0
    assert report.temperature_fahrenheit == 68.0
    assert report.condition == "Rainy"


def test_empty_city_raises_invalid_location_error():
    processor = WeatherProcessor()
    with pytest.raises(InvalidLocationError, match="City name cannot be empty."):
        processor.process_api_response("", {"temp": 15, "status": "Clear"})


def test_missing_keys_raises_external_api_error():
    processor = WeatherProcessor()
    with pytest.raises(
        ExternalAPIError, match="Required keys missing in weather payload."
    ):
        processor.process_api_response("Karachi", {"invalid_key": 100})


# Tests Run
test_process_valid_weather_data()
test_empty_city_raises_invalid_location_error()
test_missing_keys_raises_external_api_error()

print("🎉 All 3 unit tests PASSED successfully!")

ERROR:__main__:Invalid city provided.
ERROR:__main__:Missing keys in payload for city Karachi.


🎉 All 3 unit tests PASSED successfully!
